# Qwen3.8-27B inference server on Kaggle (free GPU)
Serves `ISTA-DASLab/Qwen3.8-27B-GSQ-RCO-GGUF` (IQ3_S, ~12 GB) through an OpenAI-compatible API for the support-compressor evals — a fast alternative to the NIM free tier.

**Before Run all — notebook settings (right sidebar):**
- Accelerator: **GPU T4 x2** (a single T4 or P100 also works, less headroom).
- Internet: **ON** (model + binary downloads need it).
- Persistence: not needed; everything reinstalls on re-run (~15 min).

**What success looks like:** the smoke cell prints SMOKE OK and the tunnel cell prints a `https://....trycloudflare.com` URL. Keep this notebook running while you use it (free sessions cap at ~9-12 h); the URL dies with the kernel.

In [ ]:
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    capture_output=True, text=True,
)
print(result.stdout)
assert result.returncode == 0, 'No GPU visible — enable a GPU accelerator in notebook settings (T4 x2 recommended).'
gpus = [line for line in result.stdout.strip().split('\n') if line.strip()]
print(f'detected {len(gpus)} GPU(s)')
print('--- driver / CUDA version (diagnostic) ---')
drv = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print('\n'.join(drv.stdout.split('\n')[:11]))


In [ ]:
REPO = 'ISTA-DASLab/Qwen3.8-27B-GSQ-RCO-GGUF'
GGUF_FILE = 'Qwen3.8-27B-GSQ-RCO-IQ3_S.gguf'  # ~12 GB. Alternatives in README.
N_CTX = 8192
N_GPU_LAYERS = -1  # -1 = offload all; lower (e.g. 40) on CUDA OOM.
PORT = 8080
print('config OK:', REPO, GGUF_FILE)


In [ ]:
import glob
import os
import site
import subprocess
import sys


def run(cmd, check=True):
    print('+', ' '.join(cmd), flush=True)
    return subprocess.run(cmd, check=check, capture_output=True, text=True)


def pip_install(*args):
    run([sys.executable, '-m', 'pip', 'install', '-q', *args])


pip_install('--upgrade', 'pip')
pip_install(
    'llama-cpp-python', '--extra-index-url',
    'https://abetlen.github.io/llama-cpp-python/whl/cu124',
)
pip_install('huggingface_hub', 'hf_transfer')
# Server runtime deps (llama_cpp.server needs all of these; v3 died on sse_starlette).
# Server runtime closure: upstream extra first, explicit backstop second.
# (v4 died on starlette_context; hand-enumeration misses lazy imports.)
pip_install('llama-cpp-python[server]', '--extra-index-url',
            'https://abetlen.github.io/llama-cpp-python/whl/cu124')
pip_install('sse-starlette', 'starlette-context', 'fastapi',
            'uvicorn[standard]', 'pydantic', 'pydantic-settings',
            'anyio', 'httpx')
pip_install('nvidia-cuda-runtime-cu12==12.4.*', 'nvidia-cublas-cu12==12.4.*')
sp = site.getsitepackages()[0]
libs = [f'{sp}/nvidia/cuda_runtime/lib', f'{sp}/nvidia/cublas/lib']
for d in libs:
    out = run(['ls', d])
    print(d, '->', out.stdout.strip().split(chr(10))[:6])
    assert 'libcudart' in out.stdout or 'libcublas' in out.stdout, f'missing libs in {d}'
os.environ['LD_LIBRARY_PATH'] = ':'.join(libs) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
print('LD_LIBRARY_PATH:', os.environ['LD_LIBRARY_PATH'])
so = glob.glob(f'{sp}/llama_cpp/lib/libllama.so*')
assert so, "libllama.so not found under site-packages/llama_cpp/lib"
print('--- ldd libllama (want: no not found) ---')
print(run(['ldd', so[0]], check=False).stdout[-1500:])
print('--- registering pip CUDA libs system-wide ---')
subprocess.run(['sudo', '-n', 'tee', '/etc/ld.so.conf.d/nvidia-pip.conf'],
                 input='\n'.join(libs) + '\n', check=True,
                 capture_output=True, text=True)
subprocess.run(['sudo', '-n', 'ldconfig'], check=True, capture_output=True, text=True)
print(subprocess.run('ldconfig -p | grep -m6 cudart', shell=True, capture_output=True, text=True).stdout)
# Verify the actually-used module (v3 only checked the base import).
import llama_cpp.server
print('llama_cpp.server OK (imports clean)')


In [ ]:
import os
import time

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import hf_hub_download

t0 = time.time()
GGUF_PATH = hf_hub_download(repo_id=REPO, filename=GGUF_FILE)
print(f'downloaded in {(time.time() - t0) / 60:.1f} min:', GGUF_PATH)


In [ ]:
import json
import subprocess
import time
import urllib.request

log = open('/tmp/llama-server.log', 'w')
cmd = [
    'python', '-m', 'llama_cpp.server',
    '--model', GGUF_PATH,
    '--n_gpu_layers', str(N_GPU_LAYERS),
    '--n_ctx', str(N_CTX),
    '--host', '127.0.0.1',
    '--port', str(PORT),
]
print('launch:', ' '.join(cmd))
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
for i in range(60):
    if proc.poll() is not None:
        print('--- server died on startup; log tail ---')
        print(''.join(open('/tmp/llama-server.log').readlines()[-40:]))
        raise RuntimeError(f"server exited with code {proc.returncode}")
    time.sleep(10)
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/v1/models', timeout=5) as r:
            print('READY, status', r.status)
            break
    except Exception:
        if i % 6 == 0:
            print(f'waiting for server... ({i * 10}s)')
else:
    print('--- server log tail ---')
    print(''.join(open('/tmp/llama-server.log').readlines()[-30:]))
    raise RuntimeError('server did not come up in 10 min')


In [ ]:
import json
import urllib.request


def chat(content, max_tokens=50):
    body = json.dumps({
        'messages': [{'role': 'user', 'content': content}],
        'temperature': 0,
        'max_tokens': max_tokens,
    }).encode()
    req = urllib.request.Request(
        f'http://127.0.0.1:{PORT}/v1/chat/completions',
        data=body,
        headers={'Content-Type': 'application/json'},
    )
    with urllib.request.urlopen(req, timeout=300) as r:
        return json.load(r)


d = chat('Reply with exactly: ok')
text = d['choices'][0]['message']['content']
print('reply:', repr(text[:120]))
assert text and text.strip(), 'empty reply — model misbehaving; check /tmp/llama-server.log'
assert d['choices'][0]['finish_reason'] == 'stop', d['choices'][0]['finish_reason']
print('SMOKE OK')


In [ ]:
import os
import re
import stat
import subprocess
import time
import urllib.request

CF = '/tmp/cloudflared'
if not os.path.exists(CF):
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        CF,
    )
    os.chmod(CF, os.stat(CF).st_mode | stat.S_IEXEC)
log = open('/tmp/cloudflared.log', 'w')
subprocess.Popen([CF, 'tunnel', '--url', f'http://127.0.0.1:{PORT}'], stdout=log, stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(5)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/tmp/cloudflared.log').read())
    if m:
        url = m.group(0)
        break
assert url, 'no tunnel URL in log; check /tmp/cloudflared.log'
open('/tmp/qwen-url.txt', 'w').write(url)
print('PUBLIC URL:', url)


In [ ]:
url = open('/tmp/qwen-url.txt').read().strip()
print('Give the worker this endpoint (note the /v1 suffix):')
print(f'  export QWEN_BASE_URL={url}/v1')
print('Then run evals with the Qwen endpoint (see kaggle/README.md). No API key needed; the URL itself is the secret — do not share or commit it.')


In [ ]:
import time

print('server running. Leave this cell going; stop the kernel when done (sessions cap at ~9-12 h).')
while True:
    time.sleep(600)
    print('alive — tunnel still up?', flush=True)


## Fallbacks & upgrades
- **CUDA OOM:** lower `N_CTX` to 4096, then `N_GPU_LAYERS` to 40, re-run from the launch cell.
- **P100 / single GPU:** works (IQ3_S is ~12 GB); same cells, smaller headroom.
- **Better quality:** swap `GGUF_FILE` to unsloth `Qwen3.8-27B-GGUF` Q4_K_M (~16.5 GB, needs both T4s' headroom or partial offload).
- **Faster:** `IQ2_XS` file in the same repo (~8 GB, more degradation).
- **vLLM instead:** needs T4 x2 + an AWQ quant; out of scope for this notebook, see README.
- **pip CUDA failure:** see README (cu121 index / CPU fallback).
